# 7주차 실습: 이진 탐색 트리(BST) 구현 및 성능 분석 — 학생용

## 목표·규약
재귀 탐색/삽입/삭제, 중위 정렬 검증, 삽입 순서별 높이·탐색 시간 비교.
Python 3.8+; 위에서 아래로 실행. 키는 비교 가능한 정수, 중복 삽입 무시, 없는 값 삭제는 변화 없음.
높이는 간선 기준(빈 트리 -1). 단순 BST는 균형을 보장하지 않습니다.
학생 TODO는 NotImplementedError로 즉시 중단하며 구현 후 테스트합니다.


In [1]:
import random, time, statistics
from collections import deque
print("환경 준비 완료")

환경 준비 완료


## Step 1. BSTNode
지난주 TreeNode의 data/left/right 구조를 재사용.

In [2]:
class BSTNode:
    def __init__(self, data):
        self.data = data
        self.left = None
        self.right = None
    def __repr__(self):
        return f"BSTNode({self.data!r})"

## Step 2–3. 전체 BST 클래스
재귀 결과를 `node.left/right`, 최상위 결과를 `self.root`에 재연결하세요.
삭제: 리프→None / 자식 하나→자식 / 둘→오른쪽 최소값 복사 후 후속자 삭제.
후속자는 오른쪽 자식을 가질 수 있어 재연결이 필요합니다.
하나의 클래스에 전체 메서드를 넣어 동일 이름 자기 상속을 피했습니다.
성능 실험용 반복 메서드도 포함합니다. 큰 편향 트리에 재귀 메서드를 호출하지 마세요.


In [3]:
class BinarySearchTree:
    def __init__(self):
        self.root = None

    def search(self, key):
        return self._search(self.root, key)

    def _search(self, node, key):
        if node is None:
            return False
        if key == node.data:
            return True
        if key < node.data:
            return self._search(node.left, key)
        return self._search(node.right, key)

    def insert(self, key):
        self.root = self._insert(self.root, key)

    def _insert(self, node, key):
        if node is None:
            return BSTNode(key)
        if key < node.data:
            node.left = self._insert(node.left, key)
        elif key > node.data:
            node.right = self._insert(node.right, key)
        # 같으면 중복이므로 무시
        return node

    def delete(self, key):
        self.root = self._delete(self.root, key)

    def _delete(self, node, key):
        if node is None:
            return None  # 없는 값: 변화 없음
        if key < node.data:
            node.left = self._delete(node.left, key)
        elif key > node.data:
            node.right = self._delete(node.right, key)
        else:
            # 자식 0개 또는 1개: 남은 자식(또는 None)을 부모에 연결
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left
            # 자식 2개: 오른쪽 서브트리 최소값(후속자) 복사 후 후속자 삭제
            successor = self._find_min(node.right)
            node.data = successor.data
            node.right = self._delete(node.right, successor.data)
        return node

    def _find_min(self, node):
        while node.left is not None:
            node = node.left
        return node

    def inorder(self):
        result = []
        def visit(node):
            if node is None:
                return
            visit(node.left)
            result.append(node.data)
            visit(node.right)
        visit(self.root)
        return result

    def height(self):
        def h(node):
            if node is None:
                return -1
            return 1 + max(h(node.left), h(node.right))
        return h(self.root)

    def count_nodes(self):
        def count(node):
            if node is None:
                return 0
            return 1 + count(node.left) + count(node.right)
        return count(self.root)

    def insert_iterative(self, key):
        if self.root is None:
            self.root = BSTNode(key)
            return
        node = self.root
        while True:
            if key == node.data:
                return  # 중복 종료
            if key < node.data:
                if node.left is None:
                    node.left = BSTNode(key)
                    return
                node = node.left
            else:
                if node.right is None:
                    node.right = BSTNode(key)
                    return
                node = node.right

    def search_iterative(self, key):
        node = self.root
        while node is not None:
            if key == node.data:
                return True
            node = node.left if key < node.data else node.right
        return False

    def height_iterative(self):
        if self.root is None:
            return -1
        best = 0
        stack = [(self.root, 0)]
        while stack:
            node, depth = stack.pop()
            best = max(best, depth)
            if node.left is not None:
                stack.append((node.left, depth + 1))
            if node.right is not None:
                stack.append((node.right, depth + 1))
        return best


## Step 4. 삽입/삭제 로직 검증
후속자의 오른쪽 자식, 루트 삭제, 중복, 없는 값, 무작위 혼합 연산까지 검사.

In [4]:
def make_bst(values, iterative=False):
    tree = BinarySearchTree()
    for value in values:
        if iterative: tree.insert_iterative(value)
        else: tree.insert(value)
    return tree

def assert_bst(tree, expected):
    # 조상 범위까지 검사: 바로 위 부모만 비교하는 오류 방지
    def valid(node, low=None, high=None):
        if node is None: return True
        if low is not None and node.data <= low: return False
        if high is not None and node.data >= high: return False
        return valid(node.left, low, node.data) and valid(node.right, node.data, high)
    assert valid(tree.root)
    assert tree.inorder() == sorted(set(expected))
    assert tree.count_nodes() == len(set(expected))


In [5]:
values = [50, 30, 70, 20, 40, 60, 80]
bst = make_bst(values)
assert_bst(bst, values)
assert bst.search(40) is True and bst.search(99) is False
for key in [20, 30, 70]:
    bst.delete(key)
    values.remove(key)
    assert_bst(bst, values)
    assert not bst.search(key)
    print("삭제", key, bst.inorder())
# 루트 두 자식 삭제
bst.delete(50); values.remove(50)
assert_bst(bst, values)
assert bst.root.data == 60

empty = BinarySearchTree()
empty.delete(1)
assert_bst(empty, [])
assert empty.height() == -1 and not empty.search(1)
for vals in [[5], [5,3], [5,7]]:
    t = make_bst(vals)
    t.delete(5)
    assert_bst(t, vals[1:])
# 후속자 60은 오른쪽 자식65를 갖는다: 65가 손실되면 안 됨.
t = make_bst([50,30,70,60,65,80])
t.delete(50)
assert t.root.data == 60 and t.root.right.left.data == 65
assert_bst(t, [30,60,65,70,80])
t.insert(65); t.delete(999)
assert_bst(t, [30,60,65,70,80])
# 전체 삭제로 빈 트리 복귀
for key in [30,60,65,70,80]: t.delete(key)
assert_bst(t, [])
# 고정 시드의 혼합 삽입/삭제를 Python set과 대조
rng = random.Random(7)
t, reference = BinarySearchTree(), set()
for _ in range(200):
    key = rng.randrange(50)
    if rng.random() < 0.6: t.insert(key); reference.add(key)
    else: t.delete(key); reference.discard(key)
    assert_bst(t, reference)
    assert t.search(key) == (key in reference)
print("모든 삽입·삭제·경계·불변식 테스트 통과")


삭제 20 [30, 40, 50, 60, 70, 80]
삭제 30 [40, 50, 60, 70, 80]
삭제 70 [40, 50, 60, 80]
모든 삽입·삭제·경계·불변식 테스트 통과


## Step 5. 높이 비교 실험
n=1000 재귀 삽입은 편향 트리에서 RecursionError가 날 수 있습니다. 반복 구현을 이용하며 재귀 제한을 올리지 않습니다.
최소 가능 높이 n>=1: ceil(log2(n+1))-1 = n.bit_length()-1.
무작위 트리는 기대 높이 O(log n)이지만 매번 균형이 보장되는 것은 아닙니다.


In [6]:
def build_for_experiment(n, order="random", seed=7):
    values = list(range(n))
    if order == "random":
        random.Random(seed).shuffle(values)
    elif order != "sorted":
        raise ValueError(f"알 수 없는 order: {order}")
    tree = BinarySearchTree()
    for value in values:
        tree.insert_iterative(value)
    return tree

def measure_height(n, order="random", seed=7):
    return build_for_experiment(n, order, seed).height_iterative()

n = 1000
bst_random = build_for_experiment(n, "random")
bst_sorted = build_for_experiment(n, "sorted")
assert bst_sorted.height_iterative() == n-1
print("최소 가능 높이:", n.bit_length()-1)
print("무작위/정렬 높이:", bst_random.height_iterative(), bst_sorted.height_iterative())

최소 가능 높이: 9
무작위/정렬 높이: 24 999


## Step 6. 탐색 시간 측정
동일한 키 목록, 반복 탐색 구현, 구성 시간 제외, 워밍업, 5회 중앙값. 측정값은 환경에 따라 달라집니다.

In [7]:
targets = [random.Random(17+i).randrange(n) for i in range(1000)]
def timed_search(tree, targets, repeats=5):
    for key in targets:  # 워밍업
        tree.search_iterative(key)
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        for key in targets:
            tree.search_iterative(key)
        times.append(time.perf_counter() - start)
    return statistics.median(times)
random_time = timed_search(bst_random, targets)
sorted_time = timed_search(bst_sorted, targets)
print(f"무작위 트리 탐색 중앙값: {random_time:.6f}s")
print(f"정렬 삽입 트리 중앙값: {sorted_time:.6f}s")
if random_time > 0: print(f"측정 시간비: {sorted_time/random_time:.2f}")

무작위 트리 탐색 중앙값: 0.000836s
정렬 삽입 트리 중앙값: 0.019558s
측정 시간비: 23.39


## 과제: BST 구현 및 성능 분석
1. 모든 로직 테스트 통과. 2. n=100/1000/10000의 두 삽입 순서 높이를 표로 정리.
3. 최소 높이와 비교하고 성능 영향을 3~5문장으로 분석. 시간 측정 조건도 명시.
4. 개인 GitHub Week7 폴더, 주당 최소 2회 커밋, 과제 폼 GitHub 링크 제출.
**주의:** 정렬 순서 실제 삽입은 전체 O(n²)입니다. n=10000 셀은 느릴 수 있어 마지막에 실행합니다. 체인 직접 구성은 같은 삽입 실험이 아니므로 대체하지 않습니다.


In [8]:
rows = []
for size in [100, 1000, 10000]:
    h_random = measure_height(size, "random")
    h_sorted = measure_height(size, "sorted")
    assert h_sorted == size - 1
    rows.append((size, size.bit_length()-1, h_random, h_sorted))
print("n | 최소 가능 높이 | 무작위 높이 | 정렬 높이")
for row in rows: print(*row, sep=" | ")


n | 최소 가능 높이 | 무작위 높이 | 정렬 높이
100 | 6 | 12 | 99
1000 | 9 | 24 | 999
10000 | 13 | 29 | 9999


## 분석 작성란

| n | 최소 가능 높이 | 무작위 삽입 높이 | 정렬 삽입 높이 |
|---|---|---|---|
| 100 | 6 | 12 | 99 |
| 1000 | 9 | 24 | 999 |
| 10000 | 13 | 29 | 9999 |

- **무작위/정렬 입력에서 높이:** 무작위 순열(seed=7)로 삽입하면 높이가 12 → 24 → 29로 n이 100배 늘어도 완만하게 증가해 O(log n) 경향을 보였습니다. 반면 정렬 순서로 삽입하면 모든 노드가 오른쪽 자식으로만 이어져 높이가 항상 n−1(99, 999, 9999)이 되어 사실상 연결리스트가 됩니다.
- **동일 탐색 대상의 시간 중앙값과 실험 조건:** n=1000 트리에서 같은 1000개 키(`random.Random(17+i).randrange(n)`)를 반복 탐색(`search_iterative`)으로 찾았고, 트리 구성 시간은 제외, 1회 워밍업 후 `time.perf_counter`로 5회 측정한 중앙값을 사용했습니다(Python 3.13, macOS). 무작위 트리 약 0.00069s, 정렬 삽입 트리 약 0.0217s로 약 31.5배 차이가 났습니다. 측정값은 실행 환경마다 달라질 수 있습니다.
- **편향 원인과 균형 보장 여부:** 단순 BST는 삽입 순서에 따라 모양이 정해지고 재균형 연산이 없기 때문에, 정렬된 입력은 매번 가장 큰 값이 되어 오른쪽 끝에만 붙습니다. 따라서 단순 BST는 균형을 보장하지 않으며, 최악의 경우 탐색/삽입/삭제가 O(n)이고 정렬 순서 전체 삽입은 O(n²)입니다. 균형 보장이 필요하면 AVL·레드블랙 트리 같은 자가 균형 트리가 필요합니다.
- **최소 높이와 무작위 높이가 다른 이유:** 최소 높이(⌈log₂(n+1)⌉−1)는 모든 레벨이 꽉 찬 완전 균형 트리일 때만 가능합니다. 무작위 삽입은 기대 높이가 O(log n)(약 2.99·log₂n 수준으로 수렴)이지만 상수 배가 크고, 먼저 들어온 키가 루트·상위 노드를 결정하므로 좌우 서브트리 크기가 고르게 나뉘지 않아 최소 높이의 약 2~2.7배(12 vs 6, 24 vs 9, 29 vs 13)가 나왔습니다. 그래도 정렬 삽입(999, 9999)과 비교하면 훨씬 낮아 탐색 성능에 큰 차이를 만듭니다.

## 복잡도 정리
search/insert/delete O(h+1), 재귀 보조 공간 O(h+1). 반복 탐색/삽입 공간 O(1).
중위 순회 O(n) 시간, O(h+1) 호출 스택 + O(n) 출력. 높이 계산 O(n).
균형 높이 O(log n), 최악 O(n); 서로 다른 키의 무작위 순열 삽입은 기대 O(log n) 연산.
Tree Sort 기대 O(n log n), 최악 O(n²); 중복을 보존하려면 카운터 등 정책 변경 필요.


## 선택 심화·시각화
- VisuAlgo BST 모드에서 50,30,70,20,40,60,80 삽입 후 20,30,70 삭제와 코드 결과 비교.![alt text](<스크린샷 2026-10-28 오후 5.23.35.png>)
50,30,70,20,40,60,80 삽입 후 20(리프)→30(자식 하나, 40이 올라옴)→70(자식 둘, 후속자 80으로 대체) 순으로 삭제하면 중위 결과 [40, 50, 60, 80]으로 Step 4 출력과 같습니다.

- 중위 선행자를 이용한 삭제로 변경.
자식이 둘일 때 오른쪽 최소값 대신 왼쪽 서브트리 최대값을 복사하고, 왼쪽 서브트리에서 그 값을 삭제, 선행자는 왼쪽 자식만 가질 수 있으므로 같은 방식으로 재연결합니다.

- 셔플 전처리는 기대 성능 개선이지 최악 균형 보장이 아님을 설명.
입력을 섞으면 기대 높이가 O(log n)이 될 뿐, 섞은 결과가 우연히 편향될 수 있어 최악 O(n)은 막지 못합니다. 키가 하나씩 들어오는 상황에선 미리 섞을 수도 없어 최악 보장에는 AVL·레드블랙 트리가 필요합니다.

- 자손 수 저장으로 kth_smallest 구현: O(h), 균형일 때만 O(log n). 삽입/삭제마다 size 갱신 필요.
각 노드에 서브트리 크기(size)를 저장하고, 왼쪽 크기 L과 비교해 k ≤ L이면 왼쪽, k = L+1이면 현재 노드, 아니면 k−L−1로 오른쪽에서 찾습니다. O(h)이며 균형일 때만 O(log n), 삽입/삭제 경로의 size를 모두 갱신해야 합니다.

- 다음 주 중간 평가: 연결리스트, 스택/큐, 트리, BST 복습.
